In [2]:
import os



In [ ]:
data_dir = os.path.join("../raw_dataset","1","data") #weird directory structure but this is just how the data is stored in the repo. The data is in a folder called "1" and then in a subfolder called "data".
os.listdir(data_dir)

['s25_processed',
 's26_processed',
 's4_processed',
 's14_processed',
 's27_processed',
 's22_processed',
 's20_processed',
 's31_processed',
 's30_processed',
 's2_processed',
 's17_processed',
 's13_processed',
 's5_processed',
 's1_processed',
 's11_processed',
 's32_processed',
 's33_processed',
 's18_processed',
 's16_processed',
 's10_processed',
 's7_processed',
 's15_processed',
 's9_processed',
 's3_processed',
 's34_processed',
 's12_processed',
 's23_processed',
 's8_processed',
 's6_processed',
 's24_processed',
 's29_processed',
 's28_processed',
 's19_processed']

In [ ]:
sample_path = os.path.join(data_dir,os.listdir(data_dir)[0])
files = []
dirs = []
for i in os.listdir(sample_path):
    if  os.path.isfile(os.path.join(sample_path,i)):
        files.append(i)
    else:
        dirs.append(i)
print(dirs)
print(files)  

['align']
['lwac6p.mpg', 'pbbt2n.mpg', 'bgie3s.mpg', 'srbmzp.mpg', 'pgwv7a.mpg', 'srar9s.mpg', 'pbwa7a.mpg', 'pgbv3a.mpg', 'lwap4p.mpg', 'srbs4p.mpg', 'lgax4p.mpg', 'sgbg7s.mpg', 'bgbz4p.mpg', 'bgwf7a.mpg', 'sbbl5a.mpg', 'bwwl5a.mpg', 'bgwl8n.mpg', 'bgwf4n.mpg', 'lwbx3a.mpg', 'sgba5a.mpg', 'prwozp.mpg', 'lrao7s.mpg', 'srwg1a.mpg', 'lgap9s.mpg', 'sbwszn.mpg', 'sbbz3a.mpg', 'lrwjzn.mpg', 'brby3a.mpg', 'pbwt6n.mpg', 'lrib7a.mpg', 'prwh5s.mpg', 'lwipzp.mpg', 'sril2p.mpg', 'pwbu5s.mpg', 'sbbr8p.mpg', 'pwwo5s.mpg', 'sgam9a.mpg', 'pwin5a.mpg', 'sgwn4n.mpg', 'pwbh7s.mpg', 'brbk2n.mpg', 'sgbg9a.mpg', 'bbwd6n.mpg', 'bbbq1s.mpg', 'lgwy3a.mpg', 'sgwhzn.mpg', 'pria1a.mpg', 'bwwr9a.mpg', 'lrwp5s.mpg', 'pgbo7s.mpg', 'srir7a.mpg', 'pwau1s.mpg', 'sgag4p.mpg', 'sgwa8p.mpg', 'bgwz9a.mpg', 'sbbr9a.mpg', 'lgbq2n.mpg', 'lgbd6p.mpg', 'sbwl8p.mpg', 'pgah9s.mpg', 'swwazn.mpg', 'lgaj6p.mpg', 'brwr3a.mpg', 'sbwf4p.mpg', 'srir5s.mpg', 'prwn9s.mpg', 'lrbp3a.mpg', 'sbbl3s.mpg', 'prag6n.mpg', 'bwwl2n.mpg', 'pgwc3s.m

In [ ]:
len(os.listdir(os.path.join(sample_path,dirs[0]))) == len(files)

True

So the data is stored in chunks. Each chunk is a folder that contains one directory for alignment and a lot of videos at the root of the chunk directory.

So videos at the root , their lables in align folder. 



In [8]:
sample_video_path = os.path.join(sample_path,files[0])
sample_align_path = os.path.join(sample_path,dirs[0],files[0].replace(".mpg",".align"))

import cv2 

video = cv2.VideoCapture(sample_video_path)

total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
fps = video.get(cv2.CAP_PROP_FPS)
total_time = total_frames/fps  
print(f"Total frames: {total_frames}, FPS: {fps}, Total time: {total_time}")




Total frames: 75, FPS: 25.0, Total time: 3.0


In [9]:
with open(sample_align_path,"r") as f:
    align = f.readlines()

updated_align = [align[i].strip().split() for i in range(len(align))]
updated_align



[['0', '9250', 'sil'],
 ['9250', '18250', 'lay'],
 ['18250', '27000', 'white'],
 ['27000', '28750', 'at'],
 ['28750', '36500', 'c'],
 ['36500', '46500', 'six'],
 ['46500', '58000', 'please'],
 ['58000', '74500', 'sil']]

the alignment files have three columns. the first column is the start time of the word, the second column is the end time of the word, 
and the third column is the word itself. 
 the times are in 1/25000th of a second (audio time) becasue the audio sample rate is 25KHZ. 


In [10]:

updated_align = [(int(i[0])/25000,int(i[1])/25000,i[2]) for i in updated_align]
updated_align



[(0.0, 0.37, 'sil'),
 (0.37, 0.73, 'lay'),
 (0.73, 1.08, 'white'),
 (1.08, 1.15, 'at'),
 (1.15, 1.46, 'c'),
 (1.46, 1.86, 'six'),
 (1.86, 2.32, 'please'),
 (2.32, 2.98, 'sil')]

now time to go over all data and look for outliers and stuff. 
the main purpose is to check if all videos have same video length and label length 
(the end time of the last word in the align file must be less than or equal to the video length).


In [11]:

frame_count_assumption = total_frames 

frame_counts = []
video_lengths = []
wrong_end_times = []
video_count = 0 
for chunk in os.listdir(data_dir):
    chunk_path = os.path.join(data_dir,chunk)
    for video_file in os.listdir(chunk_path):
        if video_file.endswith(".mpg"):
            video_count += 1
            video_path = os.path.join(chunk_path, video_file)
            align_path = os.path.join(chunk_path, dirs[0],video_file.replace(".mpg", ".align"))
            with open(align_path,"r") as f:
                align = f.readlines()
            updated_align = [align[i].strip().split() for i in range(len(align))]
            updated_align = [(int(i[0])/25000,int(i[1])/25000,i[2]) for i in updated_align]
            last_word_end_time = updated_align[-1][1]
            video = cv2.VideoCapture(video_path)
            total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = video.get(cv2.CAP_PROP_FPS) 
            video_lengths.append(total_frames/fps)
            frame_counts.append(total_frames)
            if last_word_end_time > total_frames/fps:
                wrong_end_times.append((video_path, last_word_end_time, total_frames/fps))
            video.release()




    





In [12]:
unique_frame_counts = set(frame_counts)
unique_video_lengths = set(video_lengths)
print(f"Unique frame counts: {unique_frame_counts}")
print(f"Unique video lengths: {unique_video_lengths}")
print(f"Wrong end times: {wrong_end_times}")

print(f"Total videos checked: {video_count}")
print(f"Total videos with wrong end times: {len(wrong_end_times)}")



Unique frame counts: {74, 75, 148}
Unique video lengths: {2.96, 3.0}
Wrong end times: [('../dataset/1/data/s25_processed/brwk8p.mpg', 2.98, 2.96), ('../dataset/1/data/s25_processed/pwin2n.mpg', 2.98, 2.96), ('../dataset/1/data/s25_processed/prat5s.mpg', 2.98, 2.96), ('../dataset/1/data/s26_processed/sgbt4s.mpg', 2.98, 2.96), ('../dataset/1/data/s26_processed/bbwx8s.mpg', 2.98, 2.96), ('../dataset/1/data/s26_processed/sbiq9p.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/pwbx5n.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/swah7n.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/pwbx6s.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/lwirzs.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/lbad5p.mpg', 2.98, 2.96), ('../dataset/1/data/s4_processed/bbaf2a.mpg', 2.98, 2.96), ('../dataset/1/data/s27_processed/bbbx4p.mpg', 2.98, 2.96), ('../dataset/1/data/s22_processed/sbws4s.mpg', 2.98, 2.96), ('../dataset/1/data/s22_processed/lrwx4a.mpg', 2.98, 2.96), ('../dataset/1/data

In [ ]:
#okay so there are 198 videos that should be removed from the dataset. the rest of the vidoes are just fine
for video_path, last_word_end_time, video_length in wrong_end_times:
    align_path = os.path.join(os.path.dirname(video_path), dirs[0], os.path.basename(video_path).replace(".mpg", ".align"))
    os.remove(video_path)
    os.remove(align_path)